In [1]:
import os
import sys

from pyspark.sql import SparkSession

sys.path.insert(0, "/tmp/batch")

from us07_rdd import (
    BRONZE_PATH,
    decode_transaction,
    filter_transactions,
    summarize_by_city,
)

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("US08-Lineage")
    .config("spark.driver.host", "jupyter")
    .config("spark.executor.cores", "2")
    .config("spark.cores.max", "4")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["MINIO_ROOT_USER"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["MINIO_ROOT_PASSWORD"])
    .getOrCreate()
)

sc = spark.sparkContext
sc.addPyFile("/tmp/batch/us07_rdd.py")

sc.setLogLevel("WARN")

print("Spark version :", spark.version)
print("Spark master :", sc.master)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/spark/.ivy2/cache
The jars for the packages stored in: /home/spark/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
io.delta#delta-spark_2.12 added as a dependency
io.graphframes#graphframes-spark3_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8d1a9e29-d39b-49bb-8460-c59861e29ef1;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.7 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.7 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.

Spark version : 3.5.7
Spark master : spark://spark-master:7077


In [2]:
lines = sc.textFile(BRONZE_PATH)

transactions = lines.map(decode_transaction).cache()

selected = filter_transactions(
    transactions,
    transaction_type="P2M",
    status="COMPLETED",
    city="Marrakech",
)

metrics = summarize_by_city(selected)

print("Les RDD ont été construits.")

26/11/08 20:20:28 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Les RDD ont été construits.


In [3]:
def show_lineage(name, rdd):
    debug = rdd.toDebugString()

    if isinstance(debug, bytes):
        debug = debug.decode("utf-8", errors="replace")

    print(f"\n===== {name} =====")
    print(debug)


show_lineage("RDD Bronze", lines)
show_lineage("RDD Transactions", transactions)
show_lineage("RDD Filtré", selected)
show_lineage("RDD Agrégé", metrics)


===== RDD Bronze =====
(2) s3a://lake/bronze/history/transactions_history.jsonl MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 []
 |  s3a://lake/bronze/history/transactions_history.jsonl HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 []

===== RDD Transactions =====
(2) PythonRDD[2] at RDD at PythonRDD.scala:53 [Memory Serialized 1x Replicated]
 |  s3a://lake/bronze/history/transactions_history.jsonl MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 [Memory Serialized 1x Replicated]
 |  s3a://lake/bronze/history/transactions_history.jsonl HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 [Memory Serialized 1x Replicated]

===== RDD Filtré =====
(2) PythonRDD[7] at RDD at PythonRDD.scala:53 []
 |  PythonRDD[2] at RDD at PythonRDD.scala:53 []
 |  s3a://lake/bronze/history/transactions_history.jsonl MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 []
 |  s3a://lake/bronze/history/transactions_history.jsonl Hado

In [4]:
count = selected.count()
results = metrics.collect()

print("Transactions retenues :", count)
print("Montants par ville :", results)

assert count == 1919
assert len(results) == 1
assert str(results[0][1][1]) == "4755142.96"

print("VALIDATION OK")

[Stage 2:>                                                          (0 + 2) / 2]

Transactions retenues : 1919
Montants par ville : [('Marrakech', (1919, Decimal('4755142.96'), Decimal('2477.93')))]
VALIDATION OK


In [5]:
show_lineage("Transactions après actions", transactions)
show_lineage("Agrégation après actions", metrics)

print("Partitions du RDD :", transactions.getNumPartitions())
print("Cache demandé :", transactions.is_cached)


===== Transactions après actions =====
(2) PythonRDD[2] at RDD at PythonRDD.scala:53 [Memory Serialized 1x Replicated]
 |       CachedPartitions: 2; MemorySize: 2.3 MiB; DiskSize: 0.0 B
 |  s3a://lake/bronze/history/transactions_history.jsonl MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 [Memory Serialized 1x Replicated]
 |  s3a://lake/bronze/history/transactions_history.jsonl HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 [Memory Serialized 1x Replicated]

===== Agrégation après actions =====
(2) PythonRDD[8] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[6] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[5] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(2) PairwiseRDD[4] at reduceByKey at /tmp/batch/us07_rdd.py:50 []
    |  PythonRDD[3] at reduceByKey at /tmp/batch/us07_rdd.py:50 []
    |  PythonRDD[2] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 2; MemorySize: 2.3 MiB; DiskSize: 0.0 B
    |  s3a://lake/bro

In [6]:
print("Application ID :", sc.applicationId)
print("Spark UI :", sc.uiWebUrl)

Application ID : app-20261108202023-0002
Spark UI : http://jupyter:4040


In [7]:
import time

assert transactions.is_cached

# Même sélection pour les deux expériences.
p2m = transactions.filter(
    lambda event: event["type"] == "P2M"
)

print("Nombre de partitions :", p2m.getNumPartitions())
print("Cache disponible :", transactions.is_cached)

Nombre de partitions : 2
Cache disponible : True


In [8]:
sc.setJobGroup(
    "US08-ST05-A",
    "US08 - Narrow : filter + map + count"
)

narrow_rdd = p2m.map(
    lambda event: (
        event["city"],
        event["amount_mad"]
    )
)

start = time.perf_counter()

narrow_count = narrow_rdd.count()

narrow_duration = time.perf_counter() - start

print("Transactions P2M :", narrow_count)
print(f"Durée narrow : {narrow_duration:.3f} s")

[Stage 3:>                                                          (0 + 2) / 2]

Transactions P2M : 32123
Durée narrow : 6.738 s


In [9]:
sc.setJobGroup(
    "US08-ST05-B",
    "US08 - Wide : reduceByKey + collect"
)

wide_rdd = summarize_by_city(p2m)

start = time.perf_counter()

wide_results = sorted(wide_rdd.collect())

wide_duration = time.perf_counter() - start

print("Résultats par ville :")

for city, (count, total, average) in wide_results:
    print(
        f"{city:<14} "
        f"count={count:<7} "
        f"total={total:.2f} MAD"
    )

assert sum(row[1][0] for row in wide_results) == narrow_count

print(f"\nDurée wide : {wide_duration:.3f} s")
print("VALIDATION OK")

[Stage 5:>                                                          (0 + 2) / 2]

Résultats par ville :
Agadir         count=3200    total=8081239.62 MAD
Casablanca     count=3210    total=8044452.18 MAD
Fes            count=3120    total=7819620.62 MAD
Kenitra        count=3209    total=8208932.72 MAD
Marrakech      count=3257    total=8108666.57 MAD
Meknes         count=3219    total=8101706.23 MAD
Oujda          count=3237    total=8040479.17 MAD
Rabat          count=3229    total=7993157.92 MAD
Tanger         count=3298    total=8218379.28 MAD
Tetouan        count=3144    total=7911931.32 MAD

Durée wide : 2.462 s
VALIDATION OK


In [10]:
from operator import add
import time

pairs = p2m.map(
    lambda event: (
        event["city"],
        event["amount_mad"]
    )
)

sc.setJobGroup(
    "US08-ST05-REDUCE",
    "US08 - reduceByKey comparison"
)

start = time.perf_counter()

reduce_results = dict(
    pairs.reduceByKey(add).collect()
)

reduce_duration = time.perf_counter() - start

print("reduceByKey :", reduce_results)
print(f"Durée : {reduce_duration:.3f} s")

[Stage 6:>                                                          (0 + 2) / 2]

reduceByKey : {'Marrakech': Decimal('8108666.57'), 'Casablanca': Decimal('8044452.18'), 'Tetouan': Decimal('7911931.32'), 'Fes': Decimal('7819620.62'), 'Tanger': Decimal('8218379.28'), 'Meknes': Decimal('8101706.23'), 'Kenitra': Decimal('8208932.72'), 'Agadir': Decimal('8081239.62'), 'Rabat': Decimal('7993157.92'), 'Oujda': Decimal('8040479.17')}
Durée : 1.800 s


In [11]:
from decimal import Decimal

sc.setJobGroup(
    "US08-ST05-GROUP",
    "US08 - groupByKey comparison"
)

start = time.perf_counter()

group_results = dict(
    pairs.groupByKey()
    .mapValues(lambda values: sum(values, Decimal("0.00")))
    .collect()
)

group_duration = time.perf_counter() - start

print("groupByKey :", group_results)
print(f"Durée : {group_duration:.3f} s")

assert group_results == reduce_results

print("VALIDATION OK : résultats identiques")

[Stage 8:>                                                          (0 + 2) / 2]

groupByKey : {'Marrakech': Decimal('8108666.57'), 'Casablanca': Decimal('8044452.18'), 'Tetouan': Decimal('7911931.32'), 'Fes': Decimal('7819620.62'), 'Tanger': Decimal('8218379.28'), 'Meknes': Decimal('8101706.23'), 'Kenitra': Decimal('8208932.72'), 'Agadir': Decimal('8081239.62'), 'Rabat': Decimal('7993157.92'), 'Oujda': Decimal('8040479.17')}
Durée : 1.206 s
VALIDATION OK : résultats identiques
